# Nguyen et al. (2018) CNN-LSTM on the 80 splits

Re-implementation of V.-H. Nguyen et al., "NLOS Identification in WLANs Using Deep LSTM with CNN Features", *Sensors* 18(11), 4057, 2018 (`nguyen_core.py`). Variant `complex` uses their input, the complex CSI of one antenna plus an RSSI substitute, and variant `ours` uses our 52-value input. Splits, windows, epoch selection and the 0.5 threshold are those of our models.

## Step 1: Settings

`RUN_VARIANTS` selects the variants and `N_PARALLEL` the number of splits trained at the same time.

In [ ]:
import os
import json
import sys
import importlib
import itertools

import matplotlib.pyplot as plt
import numpy as np
import keras

import nguyen_core as core
importlib.reload(core)

print("Python", sys.version.split()[0], "| Keras", keras.__version__, "|", sys.executable)

DATA_DIR = os.path.abspath("../output_raw_full")
RESULTS_DIR = "results"

BASE_CONFIG = dict(core.DEFAULT_CONFIG)
BASE_CONFIG.update({"data_dir": DATA_DIR, "seed": 42, "window": 50, "stride": 10})

VARIANTS = {
    "complex":   {"csi": "complex",   "use_rssi": True,  "antenna": 0,
                  "label": "Nguyen CNN-LSTM (complex CSI + RSSI)"},
    "amplitude": {"csi": "amplitude", "use_rssi": True,  "antenna": 0,
                  "label": "Nguyen CNN-LSTM* (amplitude + RSSI)"},
    "ours":      {"csi": "ours",      "use_rssi": False,
                  "label": "Nguyen CNN-LSTM (our input)"},
}
RUN_VARIANTS = ["complex", "ours"]
PROTOCOLS = ["lopo", "l2po"]
N_PARALLEL = 2

OUR_MODELS = {"CNN-Transformer": "../CNN_Transformer/results", "CNN-LSTM": "../CNN_LSTM/results",
              "RF": "../RF/results", "SVM": "../SVM/results"}

def config_of(variant):
    cfg = dict(BASE_CONFIG)
    cfg.update({k: v for k, v in VARIANTS[variant].items() if k != "label"})
    return cfg

for v in RUN_VARIANTS:
    print(f"--- {v}: {VARIANTS[v]['label']}")
    for k, val in config_of(v).items():
        print(f"  {k:17s} {val}")

COLOR_LOS = "#2a78d6"
COLOR_NLOS = "#eb6834"
INK = "#0b0b0b"
INK_2 = "#52514e"
GRID = "#e4e3df"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": INK_2, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11, "lines.linewidth": 2,
})
os.makedirs(RESULTS_DIR, exist_ok=True)

## Step 2: Load the data of each variant

In [ ]:
LOS_CLUSTERS = ["LOS_cluster1", "LOS_cluster2", "LOS_cluster3", "LOS_cluster4"]
NLOS_CLUSTERS = ["NLOS_cluster1", "NLOS_cluster2", "NLOS_cluster3", "NLOS_cluster4", "NLOS_cluster5"]
ALL_CLUSTERS = LOS_CLUSTERS + NLOS_CLUSTERS
W, S = BASE_CONFIG["window"], BASE_CONFIG["stride"]

DATA = {}
for v in RUN_VARIANTS:
    cfg = config_of(v)
    frames, labels = core.load_cluster_frames(DATA_DIR, ALL_CLUSTERS, cfg)
    DATA[v] = (frames, labels)
    print(f"=== {v}: {frames[ALL_CLUSTERS[0]].shape[1]} values per frame")
    for c in ALL_CLUSTERS:
        n_win = (len(frames[c]) - W) // S + 1
        extra = f"  RSSI substitute {frames[c][:, -1].mean():6.2f} dB" if cfg["use_rssi"] else ""
        print(f"  {c:14s} label={labels[c]}  frames={len(frames[c]):5d}  windows={n_win:4d}{extra}")

## Step 3: Models and parameter counts

At their input size, the parameter counts must equal those of their Table 3 (18,863 and 9,679).

In [ ]:
PAPER_PARAMS = {"complex": 18863, "amplitude": 9679}
for v, nf in [("complex", 113), ("amplitude", 57)]:
    m = core.build_model(config_of(v), W, nf)
    print(f"their input size, {v:9s}: {m.count_params():6d} parameters (paper: {PAPER_PARAMS[v]:6d}, "
          f"difference {100 * (m.count_params() / PAPER_PARAMS[v] - 1):+.1f}%)")
print()
for v in RUN_VARIANTS:
    frames, _ = DATA[v]
    m = core.build_model(config_of(v), W, frames[ALL_CLUSTERS[0]].shape[1])
    print(f"Argos, {v}: {m.count_params()} parameters")
m.summary()

## Step 4: Quick check on one split (optional)

In [ ]:
QUICK_CHECK = True
if QUICK_CHECK:
    v = RUN_VARIANTS[0]
    frames, labels = DATA[v]
    split = {"name": "test=LOS3+NLOS2", "test": ["LOS_cluster3", "NLOS_cluster2"],
             "train": [c for c in ALL_CLUSTERS if c not in ("LOS_cluster3", "NLOS_cluster2")]}
    r, _ = core.train_and_evaluate(config_of(v), frames, labels, split, verbose=True)
    print({k: r[k] for k in ["accuracy", "macro_f1", "roc_auc", "recall_los", "recall_nlos", "selected_epochs",
                             "inner_cv_accuracy", "tuned_threshold", "accuracy_tuned_threshold",
                             "select_seconds", "fit_seconds"]})

## Step 5: Splits

In [ ]:
short = lambda c: c.replace("_cluster", "")

def make_splits(protocol):
    if protocol == "lopo":
        tests = [[l, n] for l in LOS_CLUSTERS for n in NLOS_CLUSTERS]
    elif protocol == "l2po":
        tests = [list(ls) + list(ns) for ls in itertools.combinations(LOS_CLUSTERS, 2)
                 for ns in itertools.combinations(NLOS_CLUSTERS, 2)]
    else:
        raise ValueError(protocol)
    return [{"name": "test=" + "+".join(short(c) for c in test),
             "train": [c for c in ALL_CLUSTERS if c not in test], "test": test} for test in tests]

SPLITS = {p: make_splits(p) for p in PROTOCOLS}
for p, splits in SPLITS.items():
    print(f"{p}: {len(splits)} splits, e.g. {splits[0]['name']} ... {splits[-1]['name']}")

## Step 6: Worker for one split

In [ ]:
import subprocess

WORKER = os.path.abspath("nguyen_core.py")
CONFIG_FILES = {}
for v in RUN_VARIANTS:
    os.makedirs(os.path.join(RESULTS_DIR, v), exist_ok=True)
    CONFIG_FILES[v] = os.path.join(RESULTS_DIR, v, "part_b_config.json")
    with open(CONFIG_FILES[v], "w") as f:
        json.dump(config_of(v), f, indent=2)
    print(v, "->", CONFIG_FILES[v])

def evaluate_split(variant, split):
    arg = json.dumps({k: split[k] for k in ("name", "train", "test")})
    env = dict(os.environ, OMP_NUM_THREADS=str(max(1, (os.cpu_count() or 2) // N_PARALLEL)))
    proc = subprocess.run([sys.executable, WORKER, "--config", CONFIG_FILES[variant], "--split", arg],
                          capture_output=True, text=True, env=env)
    lines = [l for l in proc.stdout.splitlines() if l.startswith("RESULT ")]
    if proc.returncode != 0 or not lines:
        raise RuntimeError(f"{variant} / {split['name']} failed (exit code {proc.returncode}):\n{proc.stderr[-3000:]}")
    return json.loads(lines[-1][len("RESULT "):])

## Step 7: Run all splits

Results are appended to `results/<variant>/splits_<protocol>.jsonl`, and finished splits are skipped when the cell runs again.

In [ ]:
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

def load_done(path):
    if not os.path.exists(path):
        return {}
    with open(path) as f:
        return {r["name"]: r for r in (json.loads(line) for line in f if line.strip())}

lock = threading.Lock()
for v in RUN_VARIANTS:
    for protocol, splits in SPLITS.items():
        path = os.path.join(RESULTS_DIR, v, f"splits_{protocol}.jsonl")
        done = load_done(path)
        todo = [sp for sp in splits if sp["name"] not in done]
        print(f"=== {v} / {protocol}: {len(splits)} splits ({len(done)} already done)")
        t_start = time.time()
        with ThreadPoolExecutor(max_workers=N_PARALLEL) as pool:
            futures = {pool.submit(evaluate_split, v, sp): sp for sp in todo}
            for k, fut in enumerate(as_completed(futures), 1):
                r = fut.result()
                with lock:
                    with open(path, "a") as f:
                        f.write(json.dumps(r) + "\n")
                print(f"  [{len(done) + k:2d}/{len(splits)}] {r['name']:32s} acc={r['accuracy']:.4f}  "
                      f"AUC={r['roc_auc']:.4f}  epochs={r['selected_epochs']}  ({time.time() - t_start:.0f}s)")

def load_results(name_or_variant, protocol):
    d = OUR_MODELS.get(name_or_variant, os.path.join(RESULTS_DIR, name_or_variant))
    return load_done(os.path.join(d, f"splits_{protocol}.jsonl"))

RESULTS = {v: {p: load_results(v, p) for p in PROTOCOLS} for v in RUN_VARIANTS}

## Step 8: Summary per variant

`acc @tuned` uses a threshold chosen on the test predictions, as in their Table 2, and is an optimistic upper bound.

In [ ]:
SUMMARY = {}
for v in RUN_VARIANTS:
    SUMMARY[v] = {}
    print(f"=== {VARIANTS[v]['label']}")
    for protocol in PROTOCOLS:
        res = [RESULTS[v][protocol][sp["name"]] for sp in SPLITS[protocol] if sp["name"] in RESULTS[v][protocol]]
        if not res:
            continue
        get = lambda k: np.array([r[k] for r in res])
        acc, f1, auc, bal = get("accuracy"), get("macro_f1"), get("roc_auc"), get("balanced_accuracy")
        acc_t, ep, inner = get("accuracy_tuned_threshold"), get("selected_epochs"), get("inner_cv_accuracy")
        worst = res[int(acc.argmin())]
        print(f"  {protocol} ({len(res)} splits)")
        print(f"    acc @0.5   {100 * acc.mean():6.2f} ± {100 * acc.std():5.2f}   min {100 * acc.min():6.2f}   (worst: {worst['name']})")
        print(f"    macro-F1   {100 * f1.mean():6.2f}    balanced acc {100 * bal.mean():6.2f}    ROC-AUC {auc.mean():.4f}")
        print(f"    acc @tuned {100 * acc_t.mean():6.2f}   (threshold chosen on the test set: optimistic)")
        print(f"    inner held-out acc {100 * inner.mean():6.2f}   selected epochs mean {ep.mean():.1f} ({ep.min()}–{ep.max()})")
        held = {c: [r["per_cluster_accuracy"][c] for r in res if c in r["per_cluster_accuracy"]] for c in ALL_CLUSTERS}
        print("    held-out cluster accuracy (mean / worst): " +
              ", ".join(f"{short(c)} {100 * np.mean(x):.1f}/{100 * np.min(x):.1f}" for c, x in held.items() if x))
        SUMMARY[v][protocol] = {
            "n_splits": len(res), "accuracy_mean": float(acc.mean()), "accuracy_std": float(acc.std()),
            "accuracy_min": float(acc.min()), "macro_f1_mean": float(f1.mean()), "roc_auc_mean": float(auc.mean()),
            "balanced_accuracy_mean": float(bal.mean()), "accuracy_tuned_threshold_mean": float(acc_t.mean()),
            "inner_cv_accuracy_mean": float(inner.mean()), "selected_epochs_mean": float(ep.mean()),
            "worst_split": worst["name"],
            "held_out_cluster_accuracy": {c: {"mean": float(np.mean(x)), "min": float(np.min(x)), "n": len(x)}
                                          for c, x in held.items() if x}}
    with open(os.path.join(RESULTS_DIR, v, "splits_summary.json"), "w") as f:
        json.dump({"config": config_of(v), "protocols": SUMMARY[v]}, f, indent=2)

## Step 9: Comparison with our models

Friedman test with Nemenyi critical difference and Holm-corrected Wilcoxon tests on the same 80 splits.

In [ ]:
from scipy.stats import friedmanchisquare, rankdata, wilcoxon

MODELS = list(OUR_MODELS) + [v for v in RUN_VARIANTS]
NAME = {m: (VARIANTS[m]["label"] if m in VARIANTS else m) for m in MODELS}
ALL = {m: {p: (RESULTS[m][p] if m in RESULTS else load_results(m, p)) for p in PROTOCOLS} for m in MODELS}
MODELS = [m for m in MODELS if all(ALL[m][p] for p in PROTOCOLS)]

def acc_matrix(protocols):
    names = [sp["name"] for p in protocols for sp in SPLITS[p]
             if all(sp["name"] in ALL[m][p] for m in MODELS)]
    prot = [p for p in protocols for sp in SPLITS[p] if all(sp["name"] in ALL[m][p] for m in MODELS)]
    A = np.array([[ALL[m][p][n]["accuracy"] for m in MODELS] for n, p in zip(names, prot)])
    return A, names

def metric_mean(m, protocol, key):
    return np.mean([r[key] for r in ALL[m][protocol].values()])

print(f"{'model':40s} {'LOPO acc':>15s} {'LOPO min':>9s} {'L2PO acc':>15s} {'L2PO min':>9s} {'L2PO F1':>8s} {'L2PO AUC':>9s}")
TABLE = []
for m in MODELS:
    a1 = np.array([r["accuracy"] for r in ALL[m]["lopo"].values()]) * 100
    a2 = np.array([r["accuracy"] for r in ALL[m]["l2po"].values()]) * 100
    row = {"model": NAME[m], "lopo_mean": a1.mean(), "lopo_std": a1.std(), "lopo_min": a1.min(),
           "l2po_mean": a2.mean(), "l2po_std": a2.std(), "l2po_min": a2.min(),
           "l2po_f1": 100 * metric_mean(m, "l2po", "macro_f1"),
           "l2po_auc": metric_mean(m, "l2po", "roc_auc")}
    TABLE.append(row)
    print(f"{NAME[m]:40s} {row['lopo_mean']:7.2f} ± {row['lopo_std']:5.2f} {row['lopo_min']:9.2f} "
          f"{row['l2po_mean']:7.2f} ± {row['l2po_std']:5.2f} {row['l2po_min']:9.2f} {row['l2po_f1']:8.2f} {row['l2po_auc']:9.4f}")

A, names = acc_matrix(PROTOCOLS)
N, k = A.shape
stat, p_fr = friedmanchisquare(*A.T)
ranks = np.array([rankdata(-row) for row in A]).mean(axis=0)
Q_05 = {2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949, 8: 3.031}
cd = Q_05[k] * np.sqrt(k * (k + 1) / (6 * N))
print(f"\nFriedman over {N} splits and {k} models: chi2 = {stat:.1f}, p = {p_fr:.2e};  Nemenyi CD = {cd:.3f}")
for m, r in sorted(zip(MODELS, ranks), key=lambda t: t[1]):
    print(f"  mean rank {r:.2f}  {NAME[m]}")

def holm(pvals):
    order = np.argsort(pvals)
    adj = np.empty(len(pvals))
    running = 0.0
    for i, j in enumerate(order):
        running = max(running, (len(pvals) - i) * pvals[j])
        adj[j] = min(1.0, running)
    return adj

pairs = [(a, b) for a in ["CNN-Transformer", "CNN-LSTM"] if a in MODELS for b in RUN_VARIANTS if b in MODELS]
rows, pv = [], []
for a, b in pairs:
    d = A[:, MODELS.index(a)] - A[:, MODELS.index(b)]
    nz = d[d != 0]
    p = wilcoxon(nz).pvalue if len(nz) >= 6 else float("nan")
    rk = rankdata(np.abs(nz))
    r_rb = (rk[nz > 0].sum() - rk[nz < 0].sum()) / rk.sum() if len(nz) else 0.0
    rows.append((a, b, 100 * d.mean(), (d > 0).sum(), (d < 0).sum(), (d == 0).sum(), r_rb))
    pv.append(p)
PAIRWISE = []
for (a, b, md_, w, l, t, r_rb), p, p_h in zip(rows, pv, holm(np.array(pv))):
    PAIRWISE.append({"ours": a, "nguyen": NAME[b], "mean_diff_points": md_, "wins": int(w), "losses": int(l),
                     "ties": int(t), "p": float(p), "p_holm": float(p_h), "r_rank_biserial": float(r_rb)})
    print(f"  {a:16s} vs {NAME[b]:40s} {md_:+6.2f} points  W/L/T {w}/{l}/{t}  p_Holm = {p_h:.2e}  r = {r_rb:+.2f}")

print(f"\naccuracy per held-out cluster (mean over all {N} splits that hold it out)")
print(f"  {'cluster':10s}" + "".join(f"{NAME[m][:22]:>24s}" for m in MODELS))
for c in ALL_CLUSTERS:
    vals = []
    for m in MODELS:
        x = [ALL[m][p][n]["per_cluster_accuracy"][c] for p in PROTOCOLS for n in ALL[m][p]
             if c in ALL[m][p][n]["per_cluster_accuracy"]]
        vals.append(100 * np.mean(x))
    print(f"  {short(c):10s}" + "".join(f"{x:24.2f}" for x in vals))

## Step 10: Accuracy per split

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
palette = {"CNN-Transformer": "#2a78d6", "CNN-LSTM": "#1baf7a", "RF": "#8a8985", "SVM": "#b5b4af",
           "complex": "#eb6834", "amplitude": "#f2a36f", "ours": "#c0392b"}
for ax, protocol in zip(axes, PROTOCOLS):
    for i, m in enumerate(MODELS):
        a = 100 * np.array([r["accuracy"] for r in ALL[m][protocol].values()])
        jitter = np.random.default_rng(0).uniform(-0.18, 0.18, len(a))
        ax.plot(i + jitter, a, "o", color=palette.get(m, INK_2), alpha=0.55, markersize=4)
        ax.plot([i - 0.3, i + 0.3], [a.mean()] * 2, color=INK, linewidth=2)
    ax.set_xticks(range(len(MODELS)), [NAME[m].replace(" (", "\n(") for m in MODELS], rotation=0, fontsize=8)
    ax.set_title(f"{'LOPO (20 splits)' if protocol == 'lopo' else 'L2PO (60 splits)'}")
    ax.grid(axis="x", visible=False)
axes[0].set_ylabel("Test accuracy per split (%)  — bar = mean")
fig.suptitle("Same splits, same protocol: our models vs. the method of Nguyen et al. (2018)")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "comparison_accuracy_per_split.png"), dpi=150)
plt.show()

## Step 11: Reported against re-implemented accuracy

Written to `results/comparison_table.md` and `results/comparison.json`.

In [ ]:
NGUYEN_REPORTED = {"complex": 96.53, "amplitude": 95.97}
lines = ["| Model | Data / protocol | Threshold | LOPO accuracy (%) | L2PO accuracy (%) |",
         "|---|---|---|---|---|"]
for v, acc in NGUYEN_REPORTED.items():
    lines.append(f"| {VARIANTS[v]['label']}, as reported | their building, random 70/15/15 split | tuned on their data | "
                 f"{acc:.2f} (single split) | — |")
for m in MODELS:
    row = next(t for t in TABLE if t["model"] == NAME[m])
    proto = "Argos, unseen clusters (same 80 splits)"
    lines.append(f"| {NAME[m]}{' (re-implemented)' if m in VARIANTS else ''} | {proto} | 0.5 | "
                 f"{row['lopo_mean']:.2f} ± {row['lopo_std']:.2f} | {row['l2po_mean']:.2f} ± {row['l2po_std']:.2f} |")
for v in RUN_VARIANTS:
    if v in SUMMARY and "lopo" in SUMMARY[v]:
        lines.append(f"| {VARIANTS[v]['label']} (re-implemented) | Argos, unseen clusters | tuned on the test set (optimistic) | "
                     f"{100 * SUMMARY[v]['lopo']['accuracy_tuned_threshold_mean']:.2f} | "
                     f"{100 * SUMMARY[v]['l2po']['accuracy_tuned_threshold_mean']:.2f} |")
table_md = "\n".join(lines)
print(table_md)

with open(os.path.join(RESULTS_DIR, "comparison_table.md"), "w") as f:
    f.write(table_md + "\n")
with open(os.path.join(RESULTS_DIR, "comparison.json"), "w") as f:
    json.dump({"table": TABLE, "friedman": {"chi2": float(stat), "p": float(p_fr), "n_splits": int(N),
                                            "n_models": int(k), "nemenyi_cd": float(cd),
                                            "mean_ranks": {NAME[m]: float(r) for m, r in zip(MODELS, ranks)}},
               "pairwise": PAIRWISE, "nguyen_reported": NGUYEN_REPORTED}, f, indent=2)
print("\nsaved:", os.path.join(RESULTS_DIR, "comparison_table.md"), "and comparison.json")

# Diagnostics

## Step 12: Fixed-epoch training on one split

Trains each variant for 30 epochs and records the accuracy on the training clusters and on the unseen clusters after every epoch. The unseen clusters are only monitored. `run_diagnostics.py` runs the same diagnostics without the notebook.

In [ ]:
from sklearn.metrics import roc_auc_score

DIAG_VARIANTS = [v for v in ["complex", "ours"] if v in VARIANTS]
DIAG_SPLIT = {"name": "test=LOS2+NLOS3", "test": ["LOS_cluster2", "NLOS_cluster3"],
              "train": [c for c in ALL_CLUSTERS if c not in ("LOS_cluster2", "NLOS_cluster3")]}
DIAG_EPOCHS = 30
FIG_DIR = os.path.join(RESULTS_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

def data_of(v):
    return DATA[v] if v in DATA else core.load_cluster_frames(DATA_DIR, ALL_CLUSTERS, config_of(v))

class Monitor(keras.callbacks.Callback):
    def __init__(self, Xtr, ytr, Xte, yte):
        super().__init__()
        self.d, self.rows = (Xtr, ytr, Xte, yte), []

    def on_epoch_end(self, epoch, logs=None):
        Xtr, ytr, Xte, yte = self.d
        ptr = self.model.predict(Xtr, batch_size=512, verbose=0).ravel()
        pte = self.model.predict(Xte, batch_size=512, verbose=0).ravel()
        self.rows.append({"epoch": epoch + 1,
                          "train_acc": float(((ptr >= 0.5) == ytr).mean()),
                          "unseen_acc": float(((pte >= 0.5) == yte).mean()),
                          "unseen_recall_los": float((pte[yte == 1] >= 0.5).mean()),
                          "unseen_recall_nlos": float((pte[yte == 0] < 0.5).mean()),
                          "unseen_auc": float(roc_auc_score(yte, pte))})

DIAG = {}
for v in DIAG_VARIANTS:
    cfg = config_of(v)
    frames, labels = data_of(v)
    keras.backend.clear_session()
    keras.utils.set_random_seed(cfg["seed"])
    tf_ = core.Transform(cfg, frames, DIAG_SPLIT["train"])
    Xtr, ytr, _ = core.make_sequences(frames, labels, DIAG_SPLIT["train"], tf_, W, S)
    Xte, yte, _ = core.make_sequences(frames, labels, DIAG_SPLIT["test"], tf_, W, S)
    model = core.build_model(cfg, W, Xtr.shape[2])
    mon = Monitor(Xtr, ytr, Xte, yte)
    model.fit(Xtr, ytr, epochs=DIAG_EPOCHS, batch_size=cfg["batch_size"], callbacks=[mon], verbose=0)
    DIAG[v] = {"rows": mon.rows, "los_share_train": float(ytr.mean()), "los_share_unseen": float(yte.mean())}

    print(f"=== {VARIANTS[v]['label']}  (split {DIAG_SPLIT['name']}; LOS share: train {ytr.mean():.3f}, "
          f"unseen {yte.mean():.3f})")
    print(f"  {'epoch':>5s} {'train acc':>10s} {'unseen acc':>11s} {'unseen LOS rec':>15s} {'unseen NLOS rec':>16s} {'unseen AUC':>11s}")
    for r in mon.rows:
        if r["epoch"] in (1, 2, 3, 5, 10, 15, 20, 25, 30) or r["epoch"] == len(mon.rows):
            print(f"  {r['epoch']:5d} {r['train_acc']:10.4f} {r['unseen_acc']:11.4f} {r['unseen_recall_los']:15.4f} "
                  f"{r['unseen_recall_nlos']:16.4f} {r['unseen_auc']:11.4f}")
    best_train = max(r["train_acc"] for r in mon.rows)
    best_unseen_bal = max((r["unseen_recall_los"] + r["unseen_recall_nlos"]) / 2 for r in mon.rows)
    if best_train < max(0.8, ytr.mean() + 0.1):
        verdict = "does NOT learn the training clusters -> implementation or input problem"
    elif best_unseen_bal < 0.75:
        verdict = ("learns the training clusters but does NOT transfer to unseen clusters "
                   "-> memorisation of the training environments, not a bug")
    else:
        verdict = "learns the training clusters and transfers to unseen clusters"
    DIAG[v]["verdict"] = verdict
    print(f"  best training accuracy {best_train:.4f}; best balanced accuracy on unseen clusters {best_unseen_bal:.4f}")
    print(f"  VERDICT: {verdict}\n")

fig, axes = plt.subplots(1, len(DIAG), figsize=(6.2 * len(DIAG), 4.2), sharey=True, squeeze=False)
for ax, (v, d) in zip(axes[0], DIAG.items()):
    ep = [r["epoch"] for r in d["rows"]]
    ax.plot(ep, [r["train_acc"] for r in d["rows"]], color="#2a78d6", label="Training clusters: accuracy")
    ax.plot(ep, [r["unseen_acc"] for r in d["rows"]], color="#eb6834", label="Unseen clusters: accuracy")
    ax.plot(ep, [r["unseen_recall_nlos"] for r in d["rows"]], color="#1baf7a", linestyle="--",
            label="Unseen clusters: NLOS recall")
    ax.axhline(d["los_share_unseen"], color=INK_2, linestyle=":", linewidth=1)
    ax.text(ep[-1], d["los_share_unseen"] + 0.01, "always LOS", ha="right", va="bottom", fontsize=8, color=INK_2)
    ax.set_title(VARIANTS[v]["label"], fontsize=10.5)
    ax.set_xlabel("Epoch (fixed training, no selection)")
    ax.set_ylim(-0.02, 1.02)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
axes[0][0].set_ylabel("Accuracy / recall")
axes[0][0].legend(frameon=False, fontsize=8.5, loc="lower right")
fig.suptitle(f"Diagnostic A: learning curves on split {DIAG_SPLIT['name']} (unseen clusters only monitored)", fontsize=11)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig5_diagnostic_learning_curves.png"), dpi=200)
plt.show()

## Step 13: Same-environment test

Every cluster is split in time into 70% training, 15% validation and 15% test, with early stopping on the validation accuracy as in their paper. Results are written to `results/diagnostics.json`.

In [ ]:
SEEN_MAX_EPOCHS = 100
PARTS = {"train": (0.0, 0.70), "val": (0.70, 0.85), "test": (0.85, 1.0)}

SEEN = {}
for v in DIAG_VARIANTS:
    cfg = config_of(v)
    frames, labels = data_of(v)
    keras.backend.clear_session()
    keras.utils.set_random_seed(cfg["seed"])
    part = {name: {c: frames[c][int(a * len(frames[c])):int(b * len(frames[c]))] for c in ALL_CLUSTERS}
            for name, (a, b) in PARTS.items()}
    tf_ = core.Transform(cfg, part["train"], ALL_CLUSTERS)
    X, y = {}, {}
    for name in PARTS:
        X[name], y[name], _ = core.make_sequences(part[name], labels, ALL_CLUSTERS, tf_, W, S)
    model = core.build_model(cfg, W, X["train"].shape[2])
    stop = keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True)
    h = model.fit(X["train"], y["train"], validation_data=(X["val"], y["val"]), epochs=SEEN_MAX_EPOCHS,
                  batch_size=cfg["batch_size"], callbacks=[stop], verbose=0)
    p_val = model.predict(X["val"], batch_size=512, verbose=0).ravel()
    p_te = model.predict(X["test"], batch_size=512, verbose=0).ravel()
    t_val = core.tuned_threshold(y["val"], p_val)[0]
    yt = y["test"]
    SEEN[v] = {"n_windows": {k: int(len(y[k])) for k in PARTS},
               "epochs_run": len(h.history["loss"]),
               "best_epoch": int(np.argmax(h.history["val_accuracy"])) + 1,
               "accuracy": float(((p_te >= 0.5) == yt).mean()),
               "recall_los": float((p_te[yt == 1] >= 0.5).mean()),
               "recall_nlos": float((p_te[yt == 0] < 0.5).mean()),
               "roc_auc": float(roc_auc_score(yt, p_te)),
               "threshold_from_val": float(t_val),
               "accuracy_threshold_from_val": float(((p_te >= t_val) == yt).mean()),
               "los_share_test": float(yt.mean())}
    s = SEEN[v]
    print(f"=== {VARIANTS[v]['label']}: seen environments (chronological 70/15/15 per cluster)")
    print(f"  windows train/val/test {s['n_windows']}, best epoch {s['best_epoch']} of {s['epochs_run']} run")
    print(f"  test accuracy @0.5 {100 * s['accuracy']:.2f}%  (LOS recall {100 * s['recall_los']:.1f}%, "
          f"NLOS recall {100 * s['recall_nlos']:.1f}%), AUC {s['roc_auc']:.4f}")
    print(f"  test accuracy @threshold from validation ({s['threshold_from_val']:.3f}) "
          f"{100 * s['accuracy_threshold_from_val']:.2f}%   (always-LOS: {100 * s['los_share_test']:.1f}%)\n")

def unseen_mean(v, protocol):
    res = load_results(v, protocol)
    return 100 * np.mean([r["accuracy"] for r in res.values()]) if res else float("nan")

print(f"{'variant':42s} {'seen env. (%)':>14s} {'unseen LOPO (%)':>16s} {'unseen L2PO (%)':>16s}")
print(f"{'Nguyen et al., as reported (their building)':42s} {96.53:14.2f} {'—':>16s} {'—':>16s}")
for v in DIAG_VARIANTS:
    print(f"{VARIANTS[v]['label']:42s} {100 * SEEN[v]['accuracy']:14.2f} {unseen_mean(v, 'lopo'):16.2f} "
          f"{unseen_mean(v, 'l2po'):16.2f}")

with open(os.path.join(RESULTS_DIR, "diagnostics.json"), "w") as f:
    json.dump({"diagnostic_A": {"split": DIAG_SPLIT, "epochs": DIAG_EPOCHS, "variants": DIAG},
               "diagnostic_B": {"parts": PARTS, "max_epochs": SEEN_MAX_EPOCHS, "variants": SEEN}}, f, indent=2)
print("\nsaved:", os.path.join(RESULTS_DIR, "diagnostics.json"), "and", os.path.join(FIG_DIR, "fig5_diagnostic_learning_curves.png"))